# Phase 8 — Gradio Application Integration

**Roadmap reference:** `08_Gradio_Integration.ipynb`
This notebook imports reusable code from `src/contractiq` only
(`app_service.py` + `ui.py`) — it does not depend on hidden execution
order from earlier notebooks, per the roadmap's explicit requirement.

**Design source:** `PWC_CONTRACTIQ_PROJECT_IK/design/index.html` — a 9-section
sidebar-nav workspace (Overview, Contract Library, Risk Review, Ask
ContractIQ, Document Sources, Relationships, Review Tasks, Reports,
Activity & Traces). The same 9 sections are rebuilt here as native Gradio
(`gr.Sidebar` + `gr.Tabs`), each wired to `app_service.py` — which wraps
ALREADY-VERIFIED Phase 0-7 code, not a reimplementation.

**Honest constraint:** this is a native Gradio layout, not a pixel clone
of the mockup's custom hand-written HTML/CSS/JS (Gradio can't render
arbitrary custom markup). What's actually reused is the mockup's
*information architecture* — same 9 sections, same left-sidebar navigation
pattern, a teal color accent approximating the mockup's palette.

**Per-section backend mapping (what's reused vs. genuinely new):**

| Section | Backend |
|---|---|
| Overview | Phase 0 manifest + Phase 5 assessments + Phase 6 graph + Phase 7 eval report |
| Contract Library | Phase 0 manifest (`list_documents`) |
| Risk Review | Phase 5 `Assessment` (`get_risk_review`) |
| Ask ContractIQ | Phase 3 `search()` (`ask`) |
| Document Sources | Phase 3 ChromaDB (`get_document_chunks` — one new facade method) |
| Relationships | Phase 6 `ContractKnowledgeGraph` (reused directly, not rebuilt) |
| Review Tasks | **Genuinely new** — local JSON-backed CRUD, no earlier phase built this |
| Reports | Phase 7 `generate_document_report()` (reused directly) |
| Activity & Traces | Phase 7's real `phase7_llm_calls.jsonl` log (not a fabricated "demo" notice) |

**Completion gate:** the app launches from this notebook using only
`src/contractiq` imports; every section either shows real data from an
earlier phase or is honestly labeled as new.

## 1. Load configuration

In [1]:
import sys
sys.path.append("..")

from src.contractiq import config
from src.contractiq import app_service as svc
from src.contractiq.ui import build_app

print(f"Manifest documents: {len(svc.list_documents())}")
print(f"Assessed documents: {len(svc.list_assessed_doc_ids())}")
print(f"Langfuse host: {config.LANGFUSE_HOST}")

e:\Sheriff_faang\full_end_to_end_project_implementation\IK_PWC_Agentic_AI_Project\PWC_CAPSTONE_PROJECT\IK_PWC_COURSE_CAPSTONE_PROJECTS\.contract\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Manifest documents: 36
Assessed documents: 15
Langfuse host: https://us.cloud.langfuse.com


## 2. Sanity-check the service facade before launching the UI

Every number below comes from a real saved artifact (Phase 0 manifest,
Phase 5 assessments, Phase 6 graph, Phase 7 evaluation report) — nothing
here is placeholder data for the UI to display.

In [2]:
overview = svc.get_overview()
print(f"Documents: {overview['total_documents']} total, {overview['assessed_documents']} assessed")
print(f"Risk distribution: {overview['risk_distribution']}")
print(f"Knowledge graph: {overview['graph_nodes']} nodes, {overview['graph_edges']} edges")
print(f"Citation validity: {overview['citation_validity_rate']}")
print(f"Retrieval relevance: {overview['retrieval_relevance_rate']}")

Documents: 36 total, 15 assessed
Risk distribution: {'MEDIUM': 8, 'HIGH': 7}
Knowledge graph: 50 nodes, 68 edges
Citation validity: 0.946
Retrieval relevance: 1.0


## 3. Build and launch the app

`build_app()` assembles all 9 sections; `.launch()` applies the teal theme
and starts the local Gradio server. `prevent_thread_lock=True` so this
notebook cell returns control instead of blocking forever — the server
keeps running in the background until the kernel is stopped.

In [3]:
demo = build_app()
demo.launch(
    server_name="127.0.0.1",
    server_port=7860,
    inbrowser=True,
    share=False,
    prevent_thread_lock=True,
)
print("ContractIQ is running at http://127.0.0.1:7860")

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


ContractIQ is running at http://127.0.0.1:7860


## Honest status of Phase 8 at this point

- **Built:** `app_service.py` (facade over Phases 0-7: overview stats,
  document listing, risk review, search, document chunks, knowledge-graph
  queries, report generation, activity log — plus the genuinely new Review
  Tasks CRUD store) and `ui.py` (9-section Gradio workspace: `gr.Sidebar`
  + one `gr.Tab` per section, each calling `app_service.py` directly, never
  a raw backend phase module).
- **Verified, not assumed:** every facade function was sanity-tested
  against real saved artifacts before the UI was built on top of it
  (manifest, Phase 5 assessments, Phase 6 graph, Phase 7 log); the app was
  launched and its accessibility tree inspected to confirm every section
  renders with real data, not placeholders.
- **Honest constraint:** this is a native Gradio sidebar+tabs layout, not
  a pixel clone of the mockup's custom HTML/CSS — Gradio cannot render
  arbitrary hand-written markup. The 9-section information architecture
  and teal accent color are what's actually carried over.
- **Genuinely new in this phase:** Review Tasks (local JSON-backed CRUD —
  no earlier phase built a task tracker).
- **Not built yet (Phase 9, if pursued):** a FastAPI adapter reusing the
  same `app_service.py` facade for a non-Gradio deployment surface.